# Figure 3 and Figure S1: feature importance, 2016-19 vs 2020-23

- **Paper:** Fig 3, Fig S1
- **Original notebook:** `47_shap_compare_update.ipynb`
- **Reads:**
  - `outputs/24_, 26_, 27_, 42_, 44_, 45_shap_values.pkl`
- **Writes:**
  - `outputs/figures/Figure2.tiff (= paper Fig 3)`
  - `outputs/figures/Figure6.tiff (= paper Fig S1)`

> Note: Output file names still use the old figure numbers. Rename once figure numbering is final.


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import xgboost as xgb
#from sklearn.metrics import aic
import graphviz
import PIL # not necessary but mustn't fail





from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score




In [ ]:

# pip install shap  # see requirements.txt
import shap

In [ ]:
# read data
pre_school = pd.read_pickle(f'{OUTPUT_DIR}/24_shap_values.pkl')
pre_neighbor = pd.read_pickle(f'{OUTPUT_DIR}/26_shap_values.pkl')
pre_both = pd.read_pickle(f'{OUTPUT_DIR}/27_shap_values.pkl')



post_school = pd.read_pickle(f'{OUTPUT_DIR}/42_shap_values.pkl')
post_neighbor = pd.read_pickle(f'{OUTPUT_DIR}/44_shap_values.pkl')
post_both = pd.read_pickle(f'{OUTPUT_DIR}/45_shap_values.pkl')

In [ ]:
# feature names
school_features = ["Number AP Classes","Years Teaching (mean)","Years Teaching (SD)", "Distance to Closest UC",
            "Distance to Closest CSU","Chronic Absenteeism Rate","Free and Reduced Price Meals Rate","Per Pupil Expenditures",
            "Proportion White","Proportion Male","Shannon Index"]

neighbor_features = ['Citizenship Rate','High School Student Employment Rate',
                        "English Proficiency Rate",
                        "Computer and Internet Access Rate","Median Household Income",
                        "Households Without Vehicle Access","Minority Population Percentage",
                        "Single Parenthood Rate","Uninsured Rate","No High School Diploma Rate",
                       "Unemployment Rate","Poverty Rate"][::-1]

feature1 = ['Citizenship Rate','High School Student Employment Rate',
                        "English Proficiency Rate",
                        "Computer and Internet Access Rate","Median Household Income",
                        "Households Without Vehicle Access","Minority Population Percentage",
                        "Single Parenthood Rate","Uninsured Rate","No High School Diploma Rate",
                       "Unemployment Rate","Poverty Rate"][::-1]

both_features = ['Citizenship Rate (N)','High School Student Employment Rate (N)',"English Proficiency Rate (N)",
                        "Computer and Internet Access Rate (N)","Median Household Income (N)",
                        "Households Without Vehicle Access (N)","Minority Population Percentage (N)",
                        "Single Parenthood Rate (N)","Uninsured Rate (N)","No High School Diploma Rate (N)",
                        "Unemployment Rate (N)","Poverty Rate (N)",
                        'Distance to Closest CSU (S)',"Distance to Closest UC(S)",

                        "Per Pupil Expenditures (S)","Number AP Classes (S)",
                        "FRPM Rate (S)","Chronic Absenteeism Rate (S)",
                        "Years Teaching (SD) (S)","Years Teaching (Mean) (S)"][::-1]

In [ ]:
# importances


models = ['XGB', 'RF', 'Lasso', 'CatBoost']
pre_school_importances = [np.abs(pre_school[i]).mean(0) for i in range(4)]

pre_school_importance = pd.DataFrame({
    'feature': school_features,
    **{model: importance for model, importance in zip(models, pre_school_importances)}
})

# school
a = pre_school_importance
a["mean"] = pre_school_importance.iloc[:,1:4].mean(axis = 1)

pre_school_importance.iloc[:,1:5] = pre_school_importance.iloc[:,1:5]/pre_school_importance.iloc[:,1:5].sum(axis =0 )


# neighbor

pre_neighbor_importances = [np.abs(pre_neighbor[i]).mean(0) for i in range(4)]

pre_neighbor_importance = pd.DataFrame({
    'feature': neighbor_features,
    **{model: importance for model, importance in zip(models, pre_neighbor_importances)}
})

pre_neighbor_importance.iloc[:,1:5] = pre_neighbor_importance.iloc[:,1:5]/pre_neighbor_importance.iloc[:,1:5].sum(axis =0 )

# both
pre_both_importances = [np.abs(pre_both[i]).mean(0) for i in range(4)]

pre_both_importance = pd.DataFrame({
    'feature': both_features,
    **{model: importance for model, importance in zip(models, pre_both_importances)}
})

pre_both_importance.iloc[:,1:5] = pre_both_importance.iloc[:,1:5]/pre_both_importance.iloc[:,1:5].sum(axis =0 )


In [ ]:
a

In [ ]:
# importances post


models = ['XGB', 'RF', 'Lasso', 'CatBoost']
post_school_importances = [np.abs(post_school[i]).mean(0) for i in range(4)]

post_school_importance = pd.DataFrame({
    'feature': school_features,
    **{model: importance for model, importance in zip(models, post_school_importances)}
})

b = post_school_importance
b['mean'] = post_school_importance.iloc[:,1:5].mean(axis = 1)

post_school_importance.iloc[:,1:5] = post_school_importance.iloc[:,1:5]/post_school_importance.iloc[:,1:5].sum(axis =0 )


post_neighbor_importances = [np.abs(post_neighbor[i]).mean(0) for i in range(4)]

post_neighbor_importance = pd.DataFrame({
    'feature': neighbor_features,
    **{model: importance for model, importance in zip(models, post_neighbor_importances)}
})

post_neighbor_importance['mean'] = post_neighbor_importance.iloc[:,1:5].mean(axis = 1)

# post_neighbor_importance.iloc[:,1:5] = post_neighbor_importance.iloc[:,1:5]/post_neighbor_importance.iloc[:,1:5].sum(axis =0 )


post_both_importances = [np.abs(post_both[i]).mean(0) for i in range(4)]

post_both_importance = pd.DataFrame({
    'feature': both_features,
    **{model: importance for model, importance in zip(models, post_both_importances)}
})

post_both_importance.iloc[:,1:5] = post_both_importance.iloc[:,1:5]/post_both_importance.iloc[:,1:5].sum(axis =0 )



In [ ]:
a

In [ ]:
b

In [ ]:
pre_school_importance

In [ ]:
post_both_importance

In [ ]:
school_changes = (post_school_importance.iloc[:,1:5] - pre_school_importance.iloc[:,1:5])# /pre_school_importance.iloc[:,1:5]
school_changes.insert(0,'feature_names',school_features)
school_changes['avg_change'] = school_changes.iloc[:,1:5].mean(axis=1)

school_changes

In [ ]:
neighbor_changes = (post_neighbor_importance.iloc[:,1:5] - pre_neighbor_importance.iloc[:,1:5])#/pre_neighbor_importance.iloc[:,1:5]
neighbor_changes.insert(0,'feature_names',neighbor_features)

neighbor_changes['avg_change'] = 100*neighbor_changes.iloc[:,1:5].mean(axis=1)

neighbor_changes

In [ ]:
both_changes = (post_both_importance.iloc[:,1:5] - pre_both_importance.iloc[:,1:5])#/pre_both_importance.iloc[:,1:5]
both_changes.insert(0,'feature_names',both_features)
both_changes['avg_change'] = 100*both_changes.iloc[:,1:5].mean(axis=1)

both_changes

In [ ]:
# change in feature importance
school_changes.plot(kind='bar', x='feature_names', figsize=(12, 8), title="Change in SHAP Values (Post vs Pre)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np



def plot_feature_importance_comparison(pre_importance, post_importance, features, title, figsize=(8, 8)):
    # mean importance across models and normalize
    pre_raw = pre_importance.iloc[:, 1:5]
    post_raw = post_importance.iloc[:, 1:5]


    pre_normalized = pre_raw.div(pre_raw.sum(axis=0)) * 100
    post_normalized = post_raw.div(post_raw.sum(axis=0)) * 100


    pre_mean = pre_normalized.mean(axis=1)
    post_mean = post_normalized.mean(axis=1)

    # df
    plot_data = pd.DataFrame({
        'Feature': features,
        'Pre': pre_mean,
        'Post': post_mean
    })

    # sort
    plot_data = plot_data.sort_values('Pre', ascending=True)

     # plot
    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(features)) *0.3
    width = 0.1
    ax.barh(x + width/2, plot_data['Pre'], width, label='2016-19', color='#3498db', alpha=0.7)

    ax.barh(x - width/2, plot_data['Post'], width, label='2020-23', color='#2ecc71', alpha=0.7)


    ax.set_xlabel('Contribution (%)')
    ax.set_title(title, pad=10, fontsize=14)
    ax.set_yticks(x)
    ax.set_yticklabels(plot_data['Feature'], fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)



    ax.legend(loc = 'lower right',fontsize = 16)


    ax.grid(False)#, linestyle='--', alpha=0.7, axis='x')

    # plt.tight_layout()

    plt.savefig(f"{OUTPUT_DIR}/plots/{title}.png", bbox_inches='tight',dpi=600)

# school
fig1 = plot_feature_importance_comparison(
    pre_school_importance,
    post_school_importance,
    school_features,
    'School Features Importance Comparison (Pre vs Post)'
)

# neighbor
fig2 = plot_feature_importance_comparison(
    pre_neighbor_importance,
    post_neighbor_importance,
    neighbor_features,
    'Neighborhood Features Importance Comparison (Pre vs Post)'
)

# return plot_data

plt.show()

In [ ]:
def plot_feature_importance_comparison(pre_importance, post_importance, features, title,
                                     figsize=(7, 8), output_path=None,
                                     pre_label='2016-2019', post_label='2020-2023',
                                     ax=None):
    # Mean importance and normalize
    def process_importance(df):
        raw = df.iloc[:, 1:5]
        normalized = raw.div(raw.sum(axis=0)) * 100
        return normalized.mean(axis=1)

    pre_mean = process_importance(pre_importance)
    post_mean = process_importance(post_importance)

    # Create plot data
    plot_data = pd.DataFrame({
        'Feature': features,
        'Pre': pre_mean,
        'Post': post_mean
    })

    # Round the importance values before sorting
    plot_data['Pre_rounded'] = plot_data['Pre'].round(1)  # Round to 1 decimal place
    plot_data['Post_rounded'] = plot_data['Post'].round(1)  # Round to 1 decimal place

    # Sort by rounded 'Pre' importance, then by 'Feature' to break ties
    plot_data = plot_data.sort_values(by=['Pre_rounded', 'Feature'], ascending=[True, True])

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(features)) * 0.35  # spacing
    width = 0.13  # bar width

    # Plot bars
    ax.barh(x + width/2, plot_data['Pre'], width,
            label=pre_label, color='#3498db', alpha=0.8,
            edgecolor='#2980b9', linewidth=1)
    ax.barh(x - width/2, plot_data['Post'], width,
            label=post_label, color='#2ecc71', alpha=0.8,
            edgecolor='#27ae60', linewidth=1)

    ax.set_xlabel('Relative Importance (%)', fontsize=11, labelpad=8)
    ax.set_title(title, pad=None, fontsize=17, fontweight='normal', loc='left')

    # Adjust y-axis
    ax.set_yticks(x)
    ax.set_yticklabels(plot_data['Feature'], fontsize=16)

    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    ax.grid(True, linestyle='--', alpha=0.3, axis='x')

    # Add legend
    ax.legend(loc='lower right', fontsize=12, frameon=True,
              framealpha=0.9, edgecolor='none')

    # Add labels on bars
    def add_labels(positions, values, offset):
        for pos, value in zip(positions, values):
            ax.text(value + 0.3, pos + offset, f'{value:.1f}%',
                    va='center', fontsize=12, alpha=0.8)

    add_labels(x, plot_data['Pre'], width/2)
    add_labels(x, plot_data['Post'], -width/2)

    return ax

def plot_side_by_side_comparison(pre_school, post_school, school_features,
                                pre_neighbor, post_neighbor, neighbor_features,
                                figsize=(16, 8), output_path=None):
    """
    Creates side-by-side comparison plots of school and neighborhood features.
    """
    # Create figure with two subplots side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    # Plot school features
    plot_feature_importance_comparison(
        pre_school, post_school, school_features,
        '(A) School Features\nImportance Comparison',
        ax=ax1
    )

    # Plot neighborhood features
    plot_feature_importance_comparison(
        pre_neighbor, post_neighbor, neighbor_features,
        '(B) Neighborhood Features\nImportance Comparison',
        ax=ax2
    )

    # Adjust layout with tighter spacing
    plt.tight_layout(w_pad=0)  # Reduced spacing between subplots

    # Save if path provided
    if output_path:
        plt.savefig(output_path, bbox_inches='tight', dpi=300)

    return fig


fig = plot_side_by_side_comparison(
    pre_school_importance, post_school_importance, school_features,
    pre_neighbor_importance, post_neighbor_importance, neighbor_features,
   figsize=(16, 8),  # Narrower overall figure
    output_path=f"{OUTPUT_DIR}/plots/feature_importance_comparison.png"
)
plt.savefig(f'{FIG_DIR}/Figure2.tiff',dpi=600,format="tiff", pil_kwargs={"compression": "tiff_lzw"})

plt.show()



In [ ]:
def plot_feature_importance_comparison(pre_importance, post_importance, features, title,
                                     figsize=(7, 8), output_path=None,
                                     pre_label='2016-2019', post_label='2020-2023',
                                     ax=None):
    # Mean importance and normalize
    def process_importance(df):
        raw = df.iloc[:, 1:5]
        normalized = raw.div(raw.sum(axis=0)) * 100
        return normalized.mean(axis=1)

    pre_mean = process_importance(pre_importance)
    post_mean = process_importance(post_importance)

    # Create plot data
    plot_data = pd.DataFrame({
        'Feature': features,
        'Pre': pre_mean,
        'Post': post_mean
    })

    # Round the importance values before sorting
    plot_data['Pre_rounded'] = plot_data['Pre'].round(1)  # Round to 1 decimal place
    plot_data['Post_rounded'] = plot_data['Post'].round(1)  # Round to 1 decimal place

    # Sort by rounded 'Pre' importance, then by 'Feature' to break ties
    plot_data = plot_data.sort_values(by=['Pre_rounded', 'Feature'], ascending=[True, True])

    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(features)) * 0.35  # spacing
    width = 0.13  # bar width

    # Plot bars
    ax.barh(x + width/2, plot_data['Pre'], width,
            label=pre_label, color='#3498db', alpha=0.8,
            edgecolor='#2980b9', linewidth=1)
    ax.barh(x - width/2, plot_data['Post'], width,
            label=post_label, color='#2ecc71', alpha=0.8,
            edgecolor='#27ae60', linewidth=1)

    ax.set_xlabel('Relative Importance (%)', fontsize=11, labelpad=8)
    ax.set_title(title, pad=None, fontsize=17, fontweight='normal', loc='left')

    # Adjust y-axis
    ax.set_yticks(x)
    ax.set_yticklabels(plot_data['Feature'], fontsize=16)

    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    ax.grid(True, linestyle='--', alpha=0.3, axis='x')

    # Add legend
    ax.legend(loc='lower right', fontsize=12, frameon=True,
              framealpha=0.9, edgecolor='none')

    # Add labels on bars
    def add_labels(positions, values, offset):
        for pos, value in zip(positions, values):
            ax.text(value + 0.3, pos + offset, f'{value:.1f}%',
                    va='center', fontsize=12, alpha=0.8)

    add_labels(x, plot_data['Pre'], width/2)
    add_labels(x, plot_data['Post'], -width/2)

    return ax

In [ ]:
fig = plot_side_by_side_comparison(
    pre_school_importance, post_school_importance, school_features,
    pre_neighbor_importance, post_neighbor_importance, neighbor_features,
   figsize=(16, 8),  # Narrower overall figure
    output_path=f"{OUTPUT_DIR}/plots/feature_importance_comparison.png"
)

plt.show()

In [ ]:
# both
fig3 = plot_feature_importance_comparison(
    pre_both_importance,
    post_both_importance,
    both_features,
    'Combined Features Importance Comparison (Pre vs Post)',
    figsize=(10, 14)
)

plt.savefig(f'{FIG_DIR}/Figure6.tiff',dpi=600,bbox_inches = "tight",format="tiff", pil_kwargs={"compression": "tiff_lzw"})
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def create_feature_importance_subplots(pre_importance, post_importance, features, title="", figsize=(10, 8)):
    fig, axs = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle(title, fontsize=14, y=0.95)

    axs = axs.flatten()

    models = {'XGB': 1, 'RF': 2, 'Lasso': 3, 'CatBoost': 4}

    for idx, (model_name, model_idx) in enumerate(models.items()):
        ax = axs[idx]

        # normalize
        pre_vals = pre_importance.iloc[:, model_idx]
        post_vals = post_importance.iloc[:, model_idx]

        pre_normalized = (pre_vals / pre_vals.sum()) * 100
        post_normalized = (post_vals / post_vals.sum()) * 100

        plot_data = pd.DataFrame({
            'Feature': features,
            'Pre': pre_normalized,
            'Post': post_normalized
        })


        plot_data = plot_data.sort_values('Pre', ascending=True)

        x = np.arange(len(features)) * 0.23
        width = 0.1

        ax.barh(x + width/2, plot_data['Post'], width, label='Post', color='#2ecc71', alpha=0.7)
        ax.barh(x - width/2, plot_data['Pre'], width, label='Pre', color='#3498db', alpha=0.7)


        ax.set_title(model_name, pad=10, fontsize=12)
        ax.set_xlim(0, 50)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.legend(loc = 'lower right')

        if idx % 2 == 0:
            ax.set_yticks(x)
            ax.set_yticklabels(plot_data['Feature'], fontsize=9)
        else:
            ax.set_yticks(x)
            ax.set_yticklabels(plot_data['Feature'], fontsize=9)
            ax.tick_params(left = False)

     #   ax.set_yticklabels(plot_data['Feature'], fontsize=9)

        ax.grid(True, linestyle='--', alpha=0.3, axis='x')



        if idx >= 2:
            ax.set_xlabel('Contribution (%)')

    plt.tight_layout()
    return fig


In [ ]:
# school
fig1 = create_feature_importance_subplots(
    pre_school_importance,
    post_school_importance,
    school_features,

)

# neighbor
fig2 = create_feature_importance_subplots(
    pre_neighbor_importance,
    post_neighbor_importance,
    neighbor_features,

)

# both
fig3 = create_feature_importance_subplots(
    pre_both_importance,
    post_both_importance,
    both_features,

)

plt.show()